In [1]:
# ==========================================
# Notebook 10
# Maximal Marginal Relevance
# ==========================================

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
profiles_df = pd.read_csv("../data/item_profiles.csv")

profiles_df.head()

,item_id,title,category,tags,content_profile
0,1,The Early Days of Stripe,Startup,startup fintech founders entrepreneurship vent...,Title: The Early Days of Stripe Category: Star...
1,2,Building SpaceX,Startup,startup fintech founders entrepreneurship vent...,Title: Building SpaceX Category: Startup Tags:...
2,3,AI for Healthcare,Artificial Intelligence,ai machine-learning llm deep-learning transfor...,Title: AI for Healthcare Category: Artificial ...
3,4,The Psychology of Habits,Wellness,habits productivity self-improvement,Title: The Psychology of Habits Category: Well...
4,5,Cloud Computing Fundamentals,Technology,cloud computing distributed-systems scalability,Title: Cloud Computing Fundamentals Category: ...


In [3]:
item_embeddings = np.load("../data/item_embeddings.npy")

item_embeddings.shape

(10, 384)

In [4]:
user_embeddings = np.load("../data/user_embeddings.npy")

user_embeddings.shape

(5, 384)

In [5]:
candidate_df = pd.read_csv("../data/candidate_recommendations.csv")

candidate_df.head()

,user_id,item_id,title,category,rank
0,101,10,Large Language Models,Artificial Intelligence,1
1,101,10,Large Language Models,Artificial Intelligence,2
2,101,10,Large Language Models,Artificial Intelligence,3
3,101,10,Large Language Models,Artificial Intelligence,4
4,101,10,Large Language Models,Artificial Intelligence,5


In [6]:
print("Items:", len(profiles_df))

print()

print("Candidates:", len(candidate_df))

Items: 10

Candidates: 20


In [7]:
user_id = 101

In [8]:
user_index = 0

user_vector = user_embeddings[user_index]

In [9]:
user_candidates = candidate_df[candidate_df["user_id"] == user_id]

user_candidates

,user_id,item_id,title,category,rank
0,101,10,Large Language Models,Artificial Intelligence,1
1,101,10,Large Language Models,Artificial Intelligence,2
2,101,10,Large Language Models,Artificial Intelligence,3
3,101,10,Large Language Models,Artificial Intelligence,4
4,101,10,Large Language Models,Artificial Intelligence,5


In [10]:
candidate_embeddings = []

candidate_indices = []

for item_id in user_candidates["item_id"]:

    item_idx = profiles_df[profiles_df["item_id"] == item_id].index[0]

    candidate_indices.append(item_idx)

    candidate_embeddings.append(item_embeddings[item_idx])

candidate_embeddings = np.array(candidate_embeddings)

In [11]:
relevance_scores = cosine_similarity(user_vector.reshape(1, -1), candidate_embeddings)[
    0
]

relevance_scores

array([0.24428427, 0.24428427, 0.24428427, 0.24428427, 0.24428429],
      dtype=float32)

In [12]:
baseline_df = pd.DataFrame(
    {
        "item_id": user_candidates["item_id"].values,
        "title": user_candidates["title"].values,
        "relevance": relevance_scores,
    }
)

baseline_df.sort_values(by="relevance", ascending=False)

,item_id,title,relevance
4,10,Large Language Models,0.244284
0,10,Large Language Models,0.244284
1,10,Large Language Models,0.244284
2,10,Large Language Models,0.244284
3,10,Large Language Models,0.244284


In [13]:
def maximal_marginal_relevance(
    user_vector, candidate_embeddings, candidate_titles, top_k=5, lambda_param=0.7
):

    selected = []

    candidate_indices = list(range(len(candidate_titles)))

    relevance_scores = cosine_similarity(
        user_vector.reshape(1, -1), candidate_embeddings
    )[0]

    while len(selected) < min(top_k, len(candidate_indices)):

        mmr_scores = []

        for idx in candidate_indices:

            relevance = relevance_scores[idx]

            if len(selected) == 0:

                diversity_penalty = 0

            else:

                similarities = []

                for selected_idx in selected:

                    similarity = cosine_similarity(
                        candidate_embeddings[idx].reshape(1, -1),
                        candidate_embeddings[selected_idx].reshape(1, -1),
                    )[0][0]

                    similarities.append(similarity)

                diversity_penalty = max(similarities)

            mmr_score = (
                lambda_param * relevance - (1 - lambda_param) * diversity_penalty
            )

            mmr_scores.append((idx, mmr_score))

        best_idx = max(mmr_scores, key=lambda x: x[1])[0]

        selected.append(best_idx)

        candidate_indices.remove(best_idx)

    results = []

    for rank, idx in enumerate(selected):

        results.append(
            {
                "rank": rank + 1,
                "title": candidate_titles[idx],
                "relevance": relevance_scores[idx],
            }
        )

    return pd.DataFrame(results)

In [14]:
mmr_results = maximal_marginal_relevance(
    user_vector=user_vector,
    candidate_embeddings=candidate_embeddings,
    candidate_titles=user_candidates["title"].tolist(),
    top_k=5,
    lambda_param=0.7,
)

mmr_results

,rank,title,relevance
0,1,Large Language Models,0.244284
1,2,Large Language Models,0.244284
2,3,Large Language Models,0.244284


In [15]:
print("BASELINE")

display(baseline_df.sort_values(by="relevance", ascending=False).head(5))

print()

print("MMR")

display(mmr_results)

BASELINE


,item_id,title,relevance
4,10,Large Language Models,0.244284
0,10,Large Language Models,0.244284
1,10,Large Language Models,0.244284
2,10,Large Language Models,0.244284
3,10,Large Language Models,0.244284



MMR


,rank,title,relevance
0,1,Large Language Models,0.244284
1,2,Large Language Models,0.244284
2,3,Large Language Models,0.244284


In [16]:
maximal_marginal_relevance(
    user_vector,
    candidate_embeddings,
    user_candidates["title"].tolist(),
    top_k=5,
    lambda_param=0.9,
)

,rank,title,relevance
0,1,Large Language Models,0.244284
1,2,Large Language Models,0.244284
2,3,Large Language Models,0.244284


In [17]:
maximal_marginal_relevance(
    user_vector,
    candidate_embeddings,
    user_candidates["title"].tolist(),
    top_k=5,
    lambda_param=0.5,
)

,rank,title,relevance
0,1,Large Language Models,0.244284
1,2,Large Language Models,0.244284
2,3,Large Language Models,0.244284


In [18]:
all_mmr_results = []

user_ids = sorted(candidate_df["user_id"].unique())

for user_id in user_ids:

    user_index = user_id - min(user_ids)

    user_vector = user_embeddings[user_index]

    user_candidates = candidate_df[candidate_df["user_id"] == user_id]

    candidate_embeddings = []
    candidate_titles = []

    for item_id in user_candidates["item_id"]:

        idx = profiles_df[profiles_df["item_id"] == item_id].index[0]

        candidate_embeddings.append(item_embeddings[idx])

        candidate_titles.append(profiles_df.iloc[idx]["title"])

    candidate_embeddings = np.array(candidate_embeddings)

    results = maximal_marginal_relevance(
        user_vector, candidate_embeddings, candidate_titles, top_k=5, lambda_param=0.7
    )

    results["user_id"] = user_id

    all_mmr_results.append(results)

In [19]:
mmr_df = pd.concat(all_mmr_results)

mmr_df.head()

,rank,title,relevance,user_id
0,1,Large Language Models,0.244284,101
1,2,Large Language Models,0.244284,101
2,3,Large Language Models,0.244284,101
0,1,Large Language Models,0.192814,102
1,2,Large Language Models,0.192814,102


In [20]:
mmr_df.to_csv("../data/mmr_recommendations.csv", index=False)

In [21]:
saved_df = pd.read_csv("../data/mmr_recommendations.csv")

saved_df.head()

,rank,title,relevance,user_id
0,1,Large Language Models,0.244284,101
1,2,Large Language Models,0.244284,101
2,3,Large Language Models,0.244284,101
3,1,Large Language Models,0.192814,102
4,2,Large Language Models,0.192814,102
